# Python-часть тестового задания
### Библиотека: **polars**

Polars — быстрая DataFrame-библиотека на Rust с API на Python.  
Ключевые отличия от pandas:
- Нет индексов — только колонки
- Вместо `df["col"] = ...` используется `.with_columns()`
- Фильтрация через `pl.col(...)` вместо масок
- `group_by` вместо `groupby`


## 0. Установка зависимостей

In [ ]:
# Устанавливаем нужные библиотеки (выполнить один раз)
!pip install polars holidays -q

---
## Загрузка CSV-файлов в Google Colab
Запустите ячейку ниже — появится кнопка «Выбрать файлы».  
Загрузите **orders.csv** и **product_info.csv**.


In [ ]:
from google.colab import files

# Открываем диалог загрузки файлов
uploaded = files.upload()

# Убедимся, что оба файла загружены
print("Загруженные файлы:", list(uploaded.keys()))

---
## Задание 1. Загрузка и первичный просмотр
Загружаем оба CSV в DataFrame polars и смотрим первые 5 строк каждого.  
`try_parse_dates=True` — polars автоматически распознаёт колонки с датами.


In [ ]:
import polars as pl

# Загружаем orders.csv; try_parse_dates автоматически парсит даты
orders = pl.read_csv("orders.csv", try_parse_dates=True)

# Загружаем product_info.csv
products = pl.read_csv("product_info.csv")

print("=== orders — первые 5 строк ===")
print(orders.head())

print(f"\nРазмер: {orders.shape[0]} строк, {orders.shape[1]} колонок")
print("Типы данных:")
print(orders.dtypes)

print("\n=== product_info — первые 5 строк ===")
print(products.head())

print(f"\nРазмер: {products.shape[0]} строк, {products.shape[1]} колонок")
print("Типы данных:")
print(products.dtypes)

---
## Задание 2. Фильтрация
Оставляем только заказы, где `region = "Восточный"` и `quantity > 10`.

В polars фильтрация делается через `.filter()` с выражениями `pl.col(...)`.  
Условия объединяются через `&` (И) или `|` (ИЛИ).


In [ ]:
# Фильтруем: регион "Восточный" И количество > 10
east_large = orders.filter(
    (pl.col("region") == "Восточный") & (pl.col("quantity") > 10)
)

print(f"Количество заказов (Восточный, quantity > 10): {len(east_large)}")
print()
print("Первые 5 строк:")
print(east_large.head())

---
## Задание 3. Объединение таблиц + вычисляемый столбец
Соединяем `orders` и `product_info` по ключу `product_id` (LEFT JOIN).  
Добавляем столбец `total_cost = quantity × price`.

В polars:
- `join()` вместо `merge()`
- `with_columns()` для добавления/изменения колонок
- `.alias()` задаёт имя нового столбца


In [ ]:
# LEFT JOIN: все заказы + информация о товаре
merged = orders.join(products, on="product_id", how="left")

# Добавляем вычисляемый столбец total_cost
merged = merged.with_columns(
    (pl.col("quantity") * pl.col("price")).alias("total_cost")
)

print(f"Строк в объединённой таблице: {len(merged)}")
print(f"Колонки: {merged.columns}")
print()
print("Первые 5 строк:")
print(merged.head())

---
## Задание 4. Рабочие дни между датами
Для каждого заказа считаем количество рабочих дней между `order_date` и `delivery_date`.

Используем:
- `numpy.busday_count` — считает рабочие дни, принимает список праздников
- `holidays.Russia` — официальные праздники РФ
- `map_elements` — построчное применение функции в polars (аналог `apply` в pandas)

> ⚠️ `map_elements` медленнее, чем встроенные выражения polars, но нужен  
> для логики с праздниками, которую нельзя выразить через `pl.col(...)`.


In [ ]:
import holidays
import numpy as np

# Собираем праздники РФ для всех лет из датасета
years = merged["order_date"].dt.year().unique().to_list()
ru_holidays = holidays.Russia(years=years)

def count_business_days(row: dict) -> int:
    """
    Считает рабочие дни между двумя датами с учётом праздников РФ.
    Принимает dict с ключами 'order_date' и 'delivery_date'.
    """
    start = row["order_date"]
    end   = row["delivery_date"]

    if start is None or end is None:
        return None

    # Праздники только в диапазоне между датами заказа и доставки
    holiday_dates = [
        d for d in ru_holidays.keys()
        if start <= d <= end
    ]

    return int(np.busday_count(start, end, holidays=holiday_dates))

# map_elements применяет функцию к каждой строке struct из двух колонок
merged = merged.with_columns(
    pl.struct(["order_date", "delivery_date"])
    .map_elements(count_business_days, return_dtype=pl.Int64)
    .alias("business_days")
)

print("Столбец business_days добавлен. Проверка первых 10 строк:")
print(
    merged.select(["order_id", "order_date", "delivery_date", "business_days"])
    .head(10)
)

print(f"\nМин. рабочих дней: {merged['business_days'].min()}")
print(f"Макс. рабочих дней: {merged['business_days'].max()}")
print(f"Среднее рабочих дней: {merged['business_days'].mean():.2f}")

---
## Задание 5. Агрегация по регионам
Группируем по `region` и считаем три метрики.

В polars `group_by().agg([...])` принимает список выражений —  
каждое выражение описывает одну агрегатную колонку.  
`.sort()` с `descending=True` сортирует по убыванию.


In [ ]:
region_stats = (
    merged
    .group_by("region")
    .agg([
        pl.col("business_days").mean().round(2).alias("avg_business_days_by_region"),
        pl.col("total_cost").sum().alias("total_sales_by_region"),
        pl.col("order_id").count().alias("order_count_by_region"),
    ])
    .sort("avg_business_days_by_region", descending=True)
)

print("Статистика по регионам (сортировка по среднему числу рабочих дней):")
print(region_stats)

---
## Задание 6. Итоговый вывод результатов
Выводим обе таблицы в читаемом виде.


In [ ]:
# ── Итоговая таблица по регионам ──────────────────────────────────────────
print("=" * 60)
print("  СТАТИСТИКА ПО РЕГИОНАМ")
print("=" * 60)

# Переименовываем колонки для красивого вывода
print(
    region_stats.rename({
        "region":                       "Регион",
        "avg_business_days_by_region":  "Ср. рабочих дней",
        "total_sales_by_region":        "Сумма продаж (₽)",
        "order_count_by_region":        "Кол-во заказов",
    })
)

# ── Объединённая таблица (сводка) ─────────────────────────────────────────
print()
print("=" * 60)
print("  ОБЪЕДИНЁННАЯ ТАБЛИЦА — первые 10 строк")
print("=" * 60)

print(
    merged.select([
        "order_id", "order_date", "delivery_date",
        "product_name", "category", "region",
        "quantity", "price", "total_cost", "business_days"
    ])
    .head(10)
)

# ── Общие итоги ───────────────────────────────────────────────────────────
print()
print("=" * 60)
print("  ОБЩИЕ ИТОГИ")
print("=" * 60)
print(f"  Всего заказов:           {len(merged)}")
print(f"  Общая выручка:           {merged['total_cost'].sum():,.0f} ₽")
print(f"  Средний чек:             {merged['total_cost'].mean():,.0f} ₽")
print(f"  Среднее рабочих дней:    {merged['business_days'].mean():.2f}")